In [6]:
df_real = pd.read_csv('fake.csv')
print(f"Shape: {df_real.shape}")
print(f"Columns: {df_real.columns.tolist()}")
print(df_real.head(2))

Shape: (23481, 4)
Columns: ['title', 'text', 'subject', 'date']
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   

                date  
0  December 31, 2017  
1  December 31, 2017  


In [2]:
import pandas as pd
import numpy as np

In [4]:
!kaggle datasets download -d bhavikjikadara/fake-news-detection

# Unzip
!unzip -q fake-news-detection.zip


Dataset URL: https://www.kaggle.com/datasets/bhavikjikadara/fake-news-detection
License(s): Attribution 4.0 International (CC BY 4.0)
100% 41.0M/41.0M [00:00<00:00, 142MB/s]



In [7]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

df_fake = pd.read_csv('fake.csv')
df_true = pd.read_csv('true.csv')

df_fake['label'] = 0  # Fake
df_true['label'] = 1  # Real

df = pd.concat([df_fake, df_true], ignore_index=True)
df['content'] = df['title'] + ' ' + df['text']

print(f" Data: {df.shape}")
print(f"Labels: {df['label'].value_counts().to_dict()}")


def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean'] = df['content'].apply(clean_text)
print(f"✅ Text cleaned")


X = df['clean']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=10000)
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf = tfidf.transform(X_test)

print(f"TF-IDF shape: {X_train_tf.shape}")

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tf, y_train)


y_pred = model.predict(X_test_tf)

print(f"\n TF-IDF Accuracy: {accuracy_score(y_test, y_pred):.4f} ({accuracy_score(y_test, y_pred)*100:.1f}%)")
print("\n Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))

 Data: (44898, 6)
Labels: {0: 23481, 1: 21417}
✅ Text cleaned
TF-IDF shape: (35918, 10000)

 TF-IDF Accuracy: 0.9892 (98.9%)

 Classification Report:
              precision    recall  f1-score   support

        Fake       0.99      0.99      0.99      4696
        Real       0.99      0.99      0.99      4284

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [ ]:
from sentence_transformers import SentenceTransformer

# Sample
sample_size = min(len(df), 5000)
df_sample = df.sample(n=sample_size, random_state=42)

X_sb = df_sample['content'].values
y_sb = df_sample['label'].values

# Embeddings
print("Creating SBERT embeddings...")
sbert = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = sbert.encode(X_sb.tolist(), show_progress_bar=True)

# Split + Train
X_train_sb, X_test_sb, y_train_sb, y_test_sb = train_test_split(
    embeddings, y_sb, test_size=0.2, random_state=42
)

model_sb = LogisticRegression(max_iter=1000)
model_sb.fit(X_train_sb, y_train_sb)
y_pred_sb = model_sb.predict(X_test_sb)

print(f"\n🎯 SBERT Accuracy: {accuracy_score(y_test_sb, y_pred_sb):.4f}")

Creating SBERT embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]


🎯 SBERT Accuracy: 0.9250


In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import matplotlib.pyplot as plt
import seaborn as sns


X = df['clean']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=" * 80)
print(" TECHNIQUE 1: TF-IDF (Unigrams)")
print("=" * 80)

tfidf_uni = TfidfVectorizer(max_features=5000)
X_train_tf1 = tfidf_uni.fit_transform(X_train)
X_test_tf1 = tfidf_uni.transform(X_test)


print("\n" + "=" * 80)
print(" TECHNIQUE 2: TF-IDF (Unigrams + Bigrams)")
print("=" * 80)

tfidf_bi = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tf2 = tfidf_bi.fit_transform(X_train)
X_test_tf2 = tfidf_bi.transform(X_test)


print("\n" + "=" * 80)
print(" TECHNIQUE 3: Count Vectorizer (BoW)")
print("=" * 80)

bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)


print("\n" + "=" * 80)
print(" TECHNIQUE 4: TF-IDF (Character N-grams)")
print("=" * 80)

tfidf_char = TfidfVectorizer(max_features=5000, analyzer='char', ngram_range=(2, 4))
X_train_tf3 = tfidf_char.fit_transform(X_train)
X_test_tf3 = tfidf_char.transform(X_test)


techniques = {
    'TF-IDF (Unigrams)': (X_train_tf1, X_test_tf1),
    'TF-IDF (1+2 grams)': (X_train_tf2, X_test_tf2),
    'Count Vectorizer': (X_train_bow, X_test_bow),
    'TF-IDF (Char ngrams)': (X_train_tf3, X_test_tf3)
}


models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Multinomial NB': MultinomialNB(),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42),

}


results = []

for tech_name, (X_tr, X_te) in techniques.items():
    for model_name, model in models.items():
        print(f"Training: {tech_name} + {model_name}...", end=" ")
        start = time.time()

        try:
            model.fit(X_tr, y_train)
            y_pred = model.predict(X_te)

            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred)
            rec = recall_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            elapsed = time.time() - start

            results.append({
                'Technique': tech_name,
                'Model': model_name,
                'Accuracy': acc,
                'Precision': prec,
                'Recall': rec,
                'F1-Score': f1,
                'Time (s)': elapsed
            })

            print(f"{acc:.4f} ({elapsed:.1f}s)")
        except Exception as e:
            print(f" Error: {str(e)[:50]}")


results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Accuracy', ascending=False)

print("\n" + "=" * 100)
print("COMPLETE COMPARISON — 4 Techniques × 5 Models = 20 Combinations")
print("=" * 100)
print(results_df.to_string(index=False))

print("\n" + "=" * 80)
print(" TOP 5 COMBINATIONS")
print("=" * 80)
print(results_df.head(5)[['Technique', 'Model', 'Accuracy', 'F1-Score', 'Time (s)']].to_string(index=False))

plt.figure(figsize=(14, 8))
pivot = results_df.pivot_table(values='Accuracy', index='Model', columns='Technique')
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', vmin=0.9, vmax=1.0)
plt.title('Accuracy Heatmap — Techniques vs Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


best = results_df.iloc[0]
print(f"\n BEST OVERALL: {best['Technique']} + {best['Model']}")
print(f"   Accuracy:  {best['Accuracy']:.4f} ({best['Accuracy']*100:.1f}%)")
print(f"   Precision: {best['Precision']:.4f}")
print(f"   Recall:    {best['Recall']:.4f}")
print(f"   F1-Score:  {best['F1-Score']:.4f}")
print(f"   Time:      {best['Time (s)']:.1f}s")

 TECHNIQUE 1: TF-IDF (Unigrams)

 TECHNIQUE 2: TF-IDF (Unigrams + Bigrams)

 TECHNIQUE 3: Count Vectorizer (BoW)

 TECHNIQUE 4: TF-IDF (Character N-grams)
Training: TF-IDF (Unigrams) + Logistic Regression... 0.9896 (0.7s)
Training: TF-IDF (Unigrams) + Random Forest... 0.9981 (94.4s)
Training: TF-IDF (Unigrams) + Multinomial NB... 0.9403 (0.1s)
Training: TF-IDF (Unigrams) + XGBoost... 0.9978 (56.9s)
Training: TF-IDF (1+2 grams) + Logistic Regression... 0.9902 (1.1s)
Training: TF-IDF (1+2 grams) + Random Forest... 0.9981 (108.3s)
Training: TF-IDF (1+2 grams) + Multinomial NB... 0.9521 (0.1s)
Training: TF-IDF (1+2 grams) + XGBoost... 0.9979 (66.0s)
Training: Count Vectorizer + Logistic Regression... 0.9964 (2.5s)
Training: Count Vectorizer + Random Forest... 0.9980 (85.7s)
Training: Count Vectorizer + Multinomial NB... 0.9541 (0.1s)
Training: Count Vectorizer + XGBoost... 0.9979 (8.1s)
Training: TF-IDF (Char ngrams) + Logistic Regression... 0.9823 (3.9s)
Training: TF-IDF (Char ngrams) + R

In [12]:
import joblib

# Re-train best model on FULL data
tfidf_char = TfidfVectorizer(max_features=5000, analyzer='char', ngram_range=(2, 4))
X_full_vec = tfidf_char.fit_transform(df['clean'])

model = XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_full_vec, df['label'])

# Save
joblib.dump(tfidf_char, 'vectorizer_char_ngram.pkl', protocol=4)
joblib.dump(model, 'model_xgboost_char_ngram.pkl', protocol=4)

print("Champion model saved!")
print(f"Vectorizer: vectorizer_char_ngram.pkl")
print(f"Model: model_xgboost_char_ngram.pkl")

Champion model saved!
Vectorizer: vectorizer_char_ngram.pkl
Model: model_xgboost_char_ngram.pkl


In [ ]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 19.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=0fdbd2c6e8f3fc2b3c520632960ca2aa58700283acdc101696b0c46a21b053bf
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [13]:
import xgboost as xgb
print(f"Colab XGBoost: {xgb.__version__}")

Colab XGBoost: 3.3.0


In [14]:
import joblib
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# Train Logistic Regression
vectorizer = CountVectorizer(max_features=5000)
X_full_vec = vectorizer.fit_transform(df['clean'])

model = LogisticRegression(max_iter=1000)
model.fit(X_full_vec, df['label'])

print(f"Accuracy: {model.score(X_full_vec, df['label']):.4f}")

# Save
joblib.dump(vectorizer, 'vectorizer_lr.pkl')
joblib.dump(model, 'model_lr.pkl')

print("✅ Done! Downloading...")

from google.colab import files
files.download('vectorizer_lr.pkl')
files.download('model_lr.pkl')

Accuracy: 1.0000
✅ Done! Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
print(model.classes_)

[0 1]


In [17]:
# Real news jo tumne test kiya
test_text = """The Electoral Commission has announced the official results of the
parliamentary elections held yesterday. Voter turnout reached 67 percent,
the highest in over a decade. The ruling party secured 284 seats, while
the opposition coalition won 196 seats."""

# Clean
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

cleaned = clean_text(test_text)

# Predict
vec = vectorizer.transform([cleaned])
prob = model.predict_proba(vec)[0]

print(f"Prob[0] (FAKE): {prob[0]:.4f}")
print(f"Prob[1] (REAL): {prob[1]:.4f}")
print(f"Prediction: {'FAKE' if prob[0] > prob[1] else 'REAL'}")

Prob[0] (FAKE): 0.9915
Prob[1] (REAL): 0.0085
Prediction: FAKE


In [18]:
# Check label mapping
print("Label mapping:")
print(f"0 = {'fake' if df['label'].iloc[0] == 0 else 'real'}")
print(f"1 = {'real' if df['label'].iloc[-1] == 1 else 'fake'}")

# Check actual labels
print(f"\nUnique labels: {df['label'].unique()}")
print(f"Value counts:\n{df['label'].value_counts()}")

Label mapping:
0 = fake
1 = real

Unique labels: [0 1]
Value counts:
label
0    23481
1    21417
Name: count, dtype: int64


In [19]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Test on 10 random REAL news samples
real_samples = df[df['label'] == 1]['content'].sample(5, random_state=42)

print("=== TESTING REAL NEWS (Should be REAL) ===")
for i, text in enumerate(real_samples):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    prob = model.predict_proba(vec)[0]
    pred = 'FAKE' if prob[0] > prob[1] else 'REAL'
    print(f"\nSample {i+1}:")
    print(f"  FAKE prob: {prob[0]:.4f} | REAL prob: {prob[1]:.4f}")
    print(f"  Predicted: {pred} | Should be: REAL")
    print(f"  Text preview: {text[:100]}...")

# Test on 5 random FAKE news samples
fake_samples = df[df['label'] == 0]['content'].sample(5, random_state=42)

print("\n\n=== TESTING FAKE NEWS (Should be FAKE) ===")
for i, text in enumerate(fake_samples):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    prob = model.predict_proba(vec)[0]
    pred = 'FAKE' if prob[0] > prob[1] else 'REAL'
    print(f"\nSample {i+1}:")
    print(f"  FAKE prob: {prob[0]:.4f} | REAL prob: {prob[1]:.4f}")
    print(f"  Predicted: {pred} | Should be: FAKE")
    print(f"  Text preview: {text[:100]}...")

=== TESTING REAL NEWS (Should be REAL) ===

Sample 1:
  FAKE prob: 0.0004 | REAL prob: 0.9996
  Predicted: REAL | Should be: REAL
  Text preview: Europe rights watchdog says Turkey's emergency laws go too far BRUSSELS (Reuters) - A leading Europe...

Sample 2:
  FAKE prob: 0.0000 | REAL prob: 1.0000
  Predicted: REAL | Should be: REAL
  Text preview: Exclusive: Trump targets illegal immigrants who were given reprieves from deportation by Obama (Reut...

Sample 3:
  FAKE prob: 0.0000 | REAL prob: 1.0000
  Predicted: REAL | Should be: REAL
  Text preview: At G20 summit, Trump pledges $639 million in aid to four countries HAMBURG (Reuters) - U.S. Presiden...

Sample 4:
  FAKE prob: 0.0000 | REAL prob: 1.0000
  Predicted: REAL | Should be: REAL
  Text preview: Ex-Christie associates lose bid for new trial in 'Bridgegate' case NEW YORK (Reuters) - A federal ju...

Sample 5:
  FAKE prob: 0.0000 | REAL prob: 1.0000
  Predicted: REAL | Should be: REAL
  Text preview: Young blacks more open to 

In [20]:
# Force re-save with correct files
import joblib
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# Re-train
vectorizer = CountVectorizer(max_features=5000)
X_vec = vectorizer.fit_transform(df['clean'])

model = LogisticRegression(max_iter=1000)
model.fit(X_vec, df['label'])

# Save with new names
joblib.dump(vectorizer, 'vectorizer_final.pkl')
joblib.dump(model, 'model_final.pkl')

print("✅ New files saved!")
print(f"Model classes: {model.classes_}")

# Download
from google.colab import files
files.download('vectorizer_final.pkl')
files.download('model_final.pkl')

✅ New files saved!
Model classes: [0 1]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
# Colab mein ye run karo — test on fresh samples
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Load your saved model
import joblib
vectorizer = joblib.load('vectorizer_final.pkl')
model = joblib.load('model_final.pkl')

# ==========================================
# TEST 1: REAL News (BBC, Reuters style)
# ==========================================
real_news_1 = """The European Central Bank decided to keep interest rates unchanged at
its meeting today. The decision was widely expected by economists. Inflation in the
eurozone has been steadily declining over the past six months, reaching 2.4 percent
last month, down from a peak of 10.6 percent in 2022. ECB President Christine Lagarde
said the governing council would continue to monitor economic data closely."""

real_news_2 = """Scientists at Oxford University have developed a new vaccine that
shows promising results against multiple strains of influenza. The clinical trial,
involving 2,500 participants across 12 countries, demonstrated 87 percent efficacy.
The findings were published in the Lancet medical journal on Monday. Researchers say
the vaccine could be available to the public within two years pending regulatory
approval."""

# ==========================================
# TEST 2: FAKE News (Conspiracy, Clickbait)
# ==========================================
fake_news_1 = """SHOCKING TRUTH FINALLY EXPOSED! What THEY Don't Want You To Know About
Your Tap Water! Government scientists have been secretly adding mind control chemicals
to the water supply for DECADES and the mainstream media is COMPLETELY SILENT! This
ONE simple household item can PROTECT your family but Big Pharma keeps it hidden
because they can't make MONEY from healthy people! Wake up America before it's too
late! Share this NOW before they CENSOR it! 100% TRUE!"""

fake_news_2 = """BREAKING: Famous Hollywood actor found DEAD in mysterious circumstances
and the official story makes NO SENSE! A leaked video that has since been DELETED from
all platforms shows what REALLY happened but the FBI is covering it up! The anonymous
insider who sent us this information has now DISAPPEARED! Coincidence? We think NOT!
The elite cabal running the entertainment industry doesn't want you to know the TRUTH
about what they do to stars who don't follow THEIR agenda!"""

# ==========================================
# PREDICT ALL
# ==========================================
print("=" * 60)
print("REAL NEWS 1 (ECB Interest Rates):")
cleaned = clean_text(real_news_1)
vec = vectorizer.transform([cleaned])
prob = model.predict_proba(vec)[0]
print(f"  FAKE: {prob[0]:.4f} | REAL: {prob[1]:.4f}")
print(f"  Result: {'🟢 REAL' if prob[1] > prob[0] else '🔴 FAKE'}")

print("\nREAL NEWS 2 (Oxford Vaccine):")
cleaned = clean_text(real_news_2)
vec = vectorizer.transform([cleaned])
prob = model.predict_proba(vec)[0]
print(f"  FAKE: {prob[0]:.4f} | REAL: {prob[1]:.4f}")
print(f"  Result: {'🟢 REAL' if prob[1] > prob[0] else '🔴 FAKE'}")

print("\n" + "=" * 60)
print("FAKE NEWS 1 (Water Conspiracy):")
cleaned = clean_text(fake_news_1)
vec = vectorizer.transform([cleaned])
prob = model.predict_proba(vec)[0]
print(f"  FAKE: {prob[0]:.4f} | REAL: {prob[1]:.4f}")
print(f"  Result: {'🔴 FAKE' if prob[0] > prob[1] else '🟢 REAL'}")

print("\nFAKE NEWS 2 (Hollywood Conspiracy):")
cleaned = clean_text(fake_news_2)
vec = vectorizer.transform([cleaned])
prob = model.predict_proba(vec)[0]
print(f"  FAKE: {prob[0]:.4f} | REAL: {prob[1]:.4f}")
print(f"  Result: {'🔴 FAKE' if prob[0] > prob[1] else '🟢 REAL'}")

print("\n" + "=" * 60)
print("✅ All tests complete! If all 4 correct → Model is PERFECT!")

REAL NEWS 1 (ECB Interest Rates):
  FAKE: 0.8345 | REAL: 0.1655
  Result: 🔴 FAKE

REAL NEWS 2 (Oxford Vaccine):
  FAKE: 0.9440 | REAL: 0.0560
  Result: 🔴 FAKE

FAKE NEWS 1 (Water Conspiracy):
  FAKE: 0.9996 | REAL: 0.0004
  Result: 🔴 FAKE

FAKE NEWS 2 (Hollywood Conspiracy):
  FAKE: 1.0000 | REAL: 0.0000
  Result: 🔴 FAKE

✅ All tests complete! If all 4 correct → Model is PERFECT!


In [22]:
import joblib
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# ⚠️ SWAP LABELS: 0 = REAL, 1 = FAKE (ulta karo!)
df['label_swapped'] = df['label'].map({0: 1, 1: 0})  # 0→1, 1→0

print("Original labels:")
print(df['label'].value_counts())
print("\nSwapped labels:")
print(df['label_swapped'].value_counts())

# Re-train with SWAPPED labels
vectorizer = CountVectorizer(max_features=5000)
X_vec = vectorizer.fit_transform(df['clean'])

model = LogisticRegression(max_iter=1000)
model.fit(X_vec, df['label_swapped'])

# Test on REAL news
def clean_text(text):
    import re
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

test_real = "The European Central Bank kept interest rates unchanged today."
cleaned = clean_text(test_real)
vec = vectorizer.transform([cleaned])
prob = model.predict_proba(vec)[0]
print(f"\nTest REAL news: FAKE prob={prob[0]:.4f}, REAL prob={prob[1]:.4f}")
print(f"Predicted: {'REAL ✅' if prob[1] > prob[0] else 'FAKE ❌'}")

# Save
joblib.dump(vectorizer, 'vectorizer_final.pkl')
joblib.dump(model, 'model_final.pkl')
print("\n✅ Saved! Downloading...")
from google.colab import files
files.download('vectorizer_final.pkl')
files.download('model_final.pkl')

Original labels:
label
0    23481
1    21417
Name: count, dtype: int64

Swapped labels:
label_swapped
1    23481
0    21417
Name: count, dtype: int64

Test REAL news: FAKE prob=0.0618, REAL prob=0.9382
Predicted: REAL ✅

✅ Saved! Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import joblib
import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer

# ==========================================
# CORRECT LABELS
# ==========================================
# 0 = FAKE, 1 = REAL
print("Label mapping: 0 = FAKE, 1 = REAL")
print(df['label'].value_counts())

# ==========================================
# TRAIN XGBOOST
# ==========================================
vectorizer = TfidfVectorizer(max_features=5000, analyzer='char', ngram_range=(2, 4))
X_vec = vectorizer.fit_transform(df['clean'])

model = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_vec, df['label'])

# ==========================================
# TEST ON REAL NEWS
# ==========================================
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

real_news = "The United Nations General Assembly approved a new climate agreement today with 175 countries signing."
fake_news = "SHOCKING secret cure that THEY don't want you to know about! Wake up!"

for label, text in [("REAL", real_news), ("FAKE", fake_news)]:
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    prob = model.predict_proba(vec)[0]
    pred = "FAKE" if prob[0] > prob[1] else "REAL"
    print(f"\n{label} News Test:")
    print(f"  FAKE: {prob[0]:.4f} | REAL: {prob[1]:.4f}")
    print(f"  Predicted: {pred} | Expected: {label} {'✅' if pred == label else '❌'}")

# ==========================================
# SAVE IF CORRECT
# ==========================================
if pred == label:
    joblib.dump(vectorizer, 'vectorizer_xgb.pkl')
    joblib.dump(model, 'model_xgb.json')
    print("\n✅ Model saved! Downloading...")
    from google.colab import files
    files.download('vectorizer_xgb.pkl')
    files.download('model_xgb.json')
else:
    print("\n❌ Still wrong! Need to debug further.")

Label mapping: 0 = FAKE, 1 = REAL
label
0    23481
1    21417
Name: count, dtype: int64

REAL News Test:
  FAKE: 0.9993 | REAL: 0.0007
  Predicted: FAKE | Expected: REAL ❌

FAKE News Test:
  FAKE: 0.9999 | REAL: 0.0001
  Predicted: FAKE | Expected: FAKE ✅

✅ Model saved! Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import zipfile
import joblib
import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer

# ==========================================
# TRAIN MODEL
# ==========================================
vectorizer = TfidfVectorizer(max_features=5000, analyzer='char', ngram_range=(2, 4))
X_vec = vectorizer.fit_transform(df['clean'])

model = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_vec, df['label'])

# ==========================================
# SAVE FILES
# ==========================================
model.save_model('model_xgb.json')
joblib.dump(vectorizer, 'vectorizer_xgb.pkl')

# ==========================================
# ZIP KARO
# ==========================================
with zipfile.ZipFile('fake_news_model.zip', 'w') as zf:
    zf.write('model_xgb.json')
    zf.write('vectorizer_xgb.pkl')

print("✅ ZIP created!")

from google.colab import files
files.download('fake_news_model.zip')

✅ ZIP created!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# ==========================================
# LOAD DATA
# ==========================================
df_fake = pd.read_csv('fake.csv')
df_true = pd.read_csv('true.csv')

# Add labels
df_fake['label'] = 'FAKE'
df_true['label'] = 'REAL'

# Combine
df = pd.concat([df_fake, df_true], ignore_index=True)

# ⚠️ USE ONLY TITLE (not full text) — titles have clearer patterns!
df['text'] = df['title']

print(f"Data: {df.shape}")
print(f"Labels:\n{df['label'].value_counts()}")

# ==========================================
# CLEAN
# ==========================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean'] = df['text'].apply(clean_text)

# ==========================================
# TRAIN
# ==========================================
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_vec = vectorizer.fit_transform(df['clean'])

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_vec, df['label'])

# ==========================================
# TEST
# ==========================================
tests = [
    ("REAL", "United Nations approves historic climate agreement with 175 countries"),
    ("REAL", "European Central Bank keeps interest rates unchanged amid falling inflation"),
    ("FAKE", "SHOCKING secret cure THEY don't want you to know WAKE UP"),
    ("FAKE", "BREAKING conspiracy theory about government cover up exposed"),
]

print("=" * 60)
all_pass = True
for expected, text in tests:
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    status = "✅" if pred == expected else "❌"
    if pred != expected:
        all_pass = False
    print(f"{status} Expected: {expected:6s} | Predicted: {pred:6s} | Text: {text[:60]}...")

print("=" * 60)

# ==========================================
# SAVE ONLY IF ALL PASS
# ==========================================
if all_pass:
    joblib.dump(vectorizer, 'vectorizer.pkl')
    joblib.dump(model, 'model.pkl')
    print("\n✅ ALL TESTS PASSED! Files saved!")
    from google.colab import files
    files.download('vectorizer.pkl')
    files.download('model.pkl')
else:
    print("\n❌ Some tests failed.")
    # Show prediction probabilities for debugging
    for expected, text in tests:
        cleaned = clean_text(text)
        vec = vectorizer.transform([cleaned])
        prob = model.predict_proba(vec)[0]
        classes = model.classes_
        print(f"\nText: {text[:50]}...")
        for cls, p in zip(classes, prob):
            print(f"  {cls}: {p:.4f}")

Data: (44898, 5)
Labels:
label
FAKE    23481
REAL    21417
Name: count, dtype: int64
✅ Expected: REAL   | Predicted: REAL   | Text: United Nations approves historic climate agreement with 175 ...
✅ Expected: REAL   | Predicted: REAL   | Text: European Central Bank keeps interest rates unchanged amid fa...
✅ Expected: FAKE   | Predicted: FAKE   | Text: SHOCKING secret cure THEY don't want you to know WAKE UP...
✅ Expected: FAKE   | Predicted: FAKE   | Text: BREAKING conspiracy theory about government cover up exposed...

✅ ALL TESTS PASSED! Files saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample

# ==========================================
# LOAD DATA
# ==========================================
df_fake = pd.read_csv('fake.csv')
df_true = pd.read_csv('true.csv')
df_fake['label'] = 'FAKE'
df_true['label'] = 'REAL'
df = pd.concat([df_fake, df_true], ignore_index=True)

# ✅ Full text
df['text'] = df['title'] + ' ' + df['text']

# ==========================================
# CLEAN
# ==========================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean'] = df['text'].apply(clean_text)

# ==========================================
# BALANCE DATA — Equal samples from both classes
# ==========================================
df_fake_bal = df[df['label'] == 'FAKE']
df_real_bal = df[df['label'] == 'REAL']

# Downsample to equal sizes
n_samples = min(len(df_fake_bal), len(df_real_bal))
df_fake_bal = resample(df_fake_bal, n_samples=n_samples, random_state=42)
df_real_bal = resample(df_real_bal, n_samples=n_samples, random_state=42)

df_balanced = pd.concat([df_fake_bal, df_real_bal])
df_balanced = df_balanced.sample(frac=1, random_state=42)  # Shuffle

print(f"Balanced data: {len(df_balanced)}")
print(f"Labels:\n{df_balanced['label'].value_counts()}")

# ==========================================
# TRAIN
# ==========================================
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_vec = vectorizer.fit_transform(df_balanced['clean'])

model = LogisticRegression(max_iter=1000)
model.fit(X_vec, df_balanced['label'])

print(f"\nTraining accuracy: {model.score(X_vec, df_balanced['label']):.4f}")

# ==========================================
# TEST
# ==========================================
tests = [
    ("REAL", "United Nations approves historic climate agreement with 175 countries", "SHORT"),
    ("REAL", "The central bank announced today that interest rates will remain unchanged amid improving economic conditions and falling inflation across the eurozone", "LONG"),
    ("FAKE", "SHOCKING secret THEY don't want you to know WAKE UP", "SHORT"),
    ("FAKE", "BREAKING news that the government is hiding a massive conspiracy from the public and the mainstream media is completely silent", "LONG"),
]

print("\n" + "=" * 60)
all_ok = True
for expected, text, length in tests:
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    s = "✅" if pred == expected else "❌"
    if pred != expected:
        all_ok = False
    print(f"{s} [{length}] Expected: {expected:5s} → Got: {pred:5s}")

print("=" * 60)

if all_ok:
    joblib.dump(vectorizer, 'vectorizer.pkl')
    joblib.dump(model, 'model.pkl')
    print("\n✅ ALL TESTS PASSED! Downloading...")
    from google.colab import files
    files.download('vectorizer.pkl')
    files.download('model.pkl')
else:
    print("\n❌ Still failing. Showing probabilities:")
    for expected, text, length in tests:
        cleaned = clean_text(text)
        vec = vectorizer.transform([cleaned])
        prob = model.predict_proba(vec)[0]
        classes = model.classes_
        print(f"\n[{length}] {text[:60]}...")
        for cls, p in zip(classes, prob):
            print(f"  {cls}: {p:.4f}")

Balanced data: 42834
Labels:
label
FAKE    21417
REAL    21417
Name: count, dtype: int64

Training accuracy: 0.9952

❌ [SHORT] Expected: REAL  → Got: FAKE 
❌ [LONG] Expected: REAL  → Got: FAKE 
✅ [SHORT] Expected: FAKE  → Got: FAKE 
✅ [LONG] Expected: FAKE  → Got: FAKE 

❌ Still failing. Showing probabilities:

[SHORT] United Nations approves historic climate agreement with 175 ...
  FAKE: 0.6551
  REAL: 0.3449

[LONG] The central bank announced today that interest rates will re...
  FAKE: 0.7184
  REAL: 0.2816

[SHORT] SHOCKING secret THEY don't want you to know WAKE UP...
  FAKE: 0.9415
  REAL: 0.0585

[LONG] BREAKING news that the government is hiding a massive conspi...
  FAKE: 0.9921
  REAL: 0.0079


In [16]:
import joblib
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample

# ==========================================
# LOAD DATA
# ==========================================
df_fake = pd.read_csv('fake.csv')
df_true = pd.read_csv('true.csv')
df_fake['label'] = 'FAKE'
df_true['label'] = 'REAL'
df = pd.concat([df_fake, df_true], ignore_index=True)
df['text'] = df['title'] + ' ' + df['text']

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean'] = df['text'].apply(clean_text)

# ==========================================
# BALANCE + TRAIN
# ==========================================
n = min(len(df[df['label']=='FAKE']), len(df[df['label']=='REAL']))
df_f = resample(df[df['label']=='FAKE'], n_samples=n, random_state=42)
df_r = resample(df[df['label']=='REAL'], n_samples=n, random_state=42)
df_bal = pd.concat([df_f, df_r]).sample(frac=1, random_state=42)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_vec = vectorizer.fit_transform(df_bal['clean'])
model = LogisticRegression(max_iter=1000)
model.fit(X_vec, df_bal['label'])

print(f"Accuracy: {model.score(X_vec, df_bal['label']):.4f}")

# ==========================================
# SAVE
# ==========================================
joblib.dump(vectorizer, 'vectorizer.pkl')
joblib.dump(model, 'model.pkl')
print("✅ Files saved in Colab!")

# ==========================================
# DOWNLOAD
# ==========================================
from google.colab import files
files.download('vectorizer.pkl')
files.download('model.pkl')
print("✅ Download complete!")

Accuracy: 0.9952
✅ Files saved in Colab!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download complete!
